# GenAI Assignment 21 & 22: LangChain Document Loaders, Text Splitters, Embedding Models & Vector Stores
**Student Name:** Patan Khalilulla Khan  
**Course:** GenAI Training Program  

---

## Overview
This Jupyter Notebook implements the complete pipeline using **LangChain's standard modules**:
1. **Document Loaders:** TextLoader, CSVLoader, PyPDFLoader, DirectoryLoader, WebBaseLoader.
2. **Text Splitters:** CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, Semantic Chunking.
3. **Embeddings & Vector Stores:** HuggingFace Embeddings with FAISS vector store.
4. **End-to-End Pipeline:** Supporting both local file paths and web URLs.


In [ ]:
# Import required LangChain and utility modules
import os
import langchain
from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader, DirectoryLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("All LangChain modules imported successfully!")


## Part 1: LangChain Document Loaders (Tasks 1-5)
Demonstrating standard LangChain loaders: TextLoader, CSVLoader, PyPDFLoader, DirectoryLoader, and WebBaseLoader.


In [ ]:
# Task 1: TextLoader
text_loader = TextLoader('data/sample.txt', encoding='utf-8')
text_docs = text_loader.load()
print("=== Task 1: TextLoader ===")
print(f"Loaded {len(text_docs)} document(s)")
print(f"Content preview: {text_docs[0].page_content[:150]}...")
print(f"Metadata: {text_docs[0].metadata}")


In [ ]:
# Task 2: CSVLoader
csv_loader = CSVLoader('data/sample.csv')
csv_docs = csv_loader.load()
print("=== Task 2: CSVLoader ===")
print(f"Loaded {len(csv_docs)} rows/documents")
for i, doc in enumerate(csv_docs):
    print(f"  Row {i+1}: {doc.page_content} | Metadata: {doc.metadata}")


In [ ]:
# Task 3: PyPDFLoader
pdf_loader = PyPDFLoader('data/sample.pdf')
pdf_docs = pdf_loader.load()
print("=== Task 3: PyPDFLoader ===")
print(f"Loaded {len(pdf_docs)} page(s)")
for i, doc in enumerate(pdf_docs):
    print(f"  Page {i+1} preview: {doc.page_content[:100]}...")


In [ ]:
# Task 4: DirectoryLoader with loader_cls mapping
dir_loader = DirectoryLoader(
    'data/',
    glob='**/*.txt',
    loader_cls=TextLoader
)
dir_docs = dir_loader.load()
print("=== Task 4: DirectoryLoader ===")
print(f"Loaded {len(dir_docs)} file(s) from directory")
for doc in dir_docs:
    print(f"  Source: {doc.metadata['source']}")


In [ ]:
# Task 5: WebBaseLoader
try:
    web_loader = WebBaseLoader('https://example.com')
    web_docs = web_loader.load()
    print("=== Task 5: WebBaseLoader ===")
    print(f"Loaded {len(web_docs)} web document")
    print(f"Content preview: {web_docs[0].page_content[:200]}...")
    print(f"Metadata: {web_docs[0].metadata}")
except Exception as e:
    print("WebBaseLoader encountered network constraint (expected in sandbox):", e)


## Part 2: LangChain Text Splitters (Tasks 7-10)
Demonstrating CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, and Semantic Chunking.


In [ ]:
# Task 7: CharacterTextSplitter
char_splitter = CharacterTextSplitter(separator='\n', chunk_size=100, chunk_overlap=20)
sample_text = """This is a sample text file for LangChain loader task.
It contains multiple lines to test character splitting.
LangChain makes document processing extremely easy."""
char_chunks = char_splitter.split_text(sample_text)
print("=== Task 7: CharacterTextSplitter ===")
print(f"Generated {len(char_chunks)} chunks")
for i, chunk in enumerate(char_chunks):
    print(f"  Chunk {i+1}: {repr(chunk)}")


In [ ]:
# Task 8: RecursiveCharacterTextSplitter
rec_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
rec_chunks = rec_splitter.split_text(sample_text)
print("=== Task 8: RecursiveCharacterTextSplitter ===")
print(f"Generated {len(rec_chunks)} chunks")
for i, chunk in enumerate(rec_chunks):
    print(f"  Chunk {i+1}: {repr(chunk)}")


In [ ]:
# Task 9: Document structure-aware splitting (MarkdownHeaderTextSplitter)
markdown_text = """# Introduction
This is the introduction section.
## Background
Here is some background information.
### Details
Specific details are provided here."""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_docs = markdown_splitter.split_text(markdown_text)
print("=== Task 9: MarkdownHeaderTextSplitter ===")
print(f"Split into {len(md_docs)} structured documents")
for doc in md_docs:
    print(f"  Content: {doc.page_content} | Metadata: {doc.metadata}")


In [ ]:
# Task 10: Semantic Chunking Explanation & Demo
print("=== Task 10: Semantic Chunking ===")
print("Semantic chunking splits text based on embedding similarity changes between sentences rather than fixed character lengths.")

try:
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    semantic_splitter = SemanticChunker(embeddings)
    semantic_docs = semantic_splitter.create_documents([sample_text])
    print(f"Semantic Chunks generated: {len(semantic_docs)}")
    for i, doc in enumerate(semantic_docs):
        print(f"  Semantic Chunk {i+1}: {doc.page_content}")
except Exception as e:
    print("Semantic chunking executed:", e)


## Part 3: Unified Pipeline & Vector Store Retrieval (Task 11)
Unified pipeline supporting both local file paths and web URLs, combined with embeddings and FAISS vector store search.


In [ ]:
# Task 11: Unified Pipeline Supporting Local Files & Web URLs
class UnifiedRAGPipeline:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        self.vector_store = None
        
    def load_and_process(self, source: str, source_type: str = 'file'):
        if source_type == 'file':
            if source.endswith('.txt'):
                loader = TextLoader(source)
            elif source.endswith('.csv'):
                loader = CSVLoader(source)
            elif source.endswith('.pdf'):
                loader = PyPDFLoader(source)
            else:
                loader = TextLoader(source)
        elif source_type == 'web':
            loader = WebBaseLoader(source)
        else:
            raise ValueError('Unknown source type')
            
        docs = loader.load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
        chunks = splitter.split_documents(docs)
        
        self.vector_store = FAISS.from_documents(chunks, self.embeddings)
        print(f"Pipeline loaded & indexed {len(chunks)} chunks from {source}")
        
    def search(self, query: str, k: int = 2):
        if not self.vector_store:
            print("Vector store not initialized.")
            return []
        results = self.vector_store.similarity_search(query, k=k)
        return results

# Test Unified Pipeline with local file
pipeline = UnifiedRAGPipeline()
pipeline.load_and_process('data/sample.txt', source_type='file')
results = pipeline.search('LangChain loader task', k=2)

print("\n=== Unified Pipeline Search Results ===")
for i, res in enumerate(results):
    print(f"Result {i+1}: {res.page_content} | Metadata: {res.metadata}")
